# SLM Document Analysis: Backend (Google Colab)

This notebook runs the **backend** for the *SLM Document Analysis* proof of concept. The React
frontend sends **one request containing a PDF and a question**. The backend extracts the PDF text
**with its structure kept** (headings, lists, tables as Markdown), builds a prompt in the **same format
the model was fine-tuned with**, runs your **Small Language Model (base model + LoRA adapter)**, and
returns the answer. The frontend on your MacBook reaches it over HTTPS through a temporary tunnel.

**Inference only.** Nothing is trained here. The LoRA adapter is an already-trained artefact that is
only *loaded*. There is **no chunking, no retrieval (no RAG) and no server-side storage**: the whole
document goes into the prompt and nothing is kept after the response is sent.

**How to run**
1. `Runtime -> Change runtime type -> GPU` (a T4 is plenty for a 0.5B model).
2. Fill in **Cell 2** (model and adapter). No model is assumed: you must supply it.
3. Run the cells **in order**, 1 to 8. Cell 8 prints the URL that goes into the frontend's `.env`.

**Limitations (read this)**
* Text-based PDFs only. **There is no OCR**: scanned PDFs are rejected.
* Structure detection is heuristic (font size, bold, ruling lines). Tables without ruling lines are
  read as plain text.
* The adapter was trained on examples of at most 384 tokens. Whole multi-page documents are far
  longer, so answer quality on them must be evaluated; the training set does not establish it.
* If a document exceeds `MAX_INPUT_TOKENS`, the END of the document is cut and the response says so.
* The tunnel URL changes every time you start it. Update the frontend `.env` and restart `npm run dev`.
* Answers come from a small model and can be wrong.

> The Python in this notebook is generated from the unit-tested backend modules in
> `slm-document-analysis-backend/app/` by `colab/build_notebook.py`. If you change the backend
> source, rebuild the notebook instead of editing both.


## Cell 1: Install dependencies

Only what the code imports is installed. **PyTorch is not reinstalled**: Colab already ships a
CUDA build of it, and replacing it would be slow and could break the GPU runtime.

* `fastapi`, `uvicorn`, `python-multipart`: the API server and multipart (file upload) parsing
* `pydantic`, `pydantic-settings`: request validation and the settings object
* `pymupdf`: PDF text extraction
* `transformers`, `peft`: loading the base model and the LoRA adapter

If pip upgrades `transformers` or `pydantic` and a later cell raises an import error, use
`Runtime -> Restart session` and run the cells again from Cell 1.


In [ ]:
%pip install -q "fastapi>=0.115" "uvicorn>=0.30" "python-multipart>=0.0.9" "pydantic>=2.7" "pydantic-settings>=2.3" "pymupdf>=1.24" "transformers>=4.56" "peft>=0.13"

import importlib.metadata as _md

for _pkg in ["fastapi", "uvicorn", "pydantic", "pydantic-settings", "pymupdf", "torch", "transformers", "peft"]:
    print(f"{_pkg:<18} {_md.version(_pkg)}")


## Cell 2: Configuration

Edit the values in the **"EDIT THESE VALUES"** block. Everything below it is the settings class
(taken from the tested backend) and normally needs no changes.

**Values you must supply**

| Variable | Meaning |
|---|---|
| `MODEL_NAME_OR_PATH` | The **same base model** you fine-tuned: a Hugging Face id (for example `"org/model-name"`) or a path such as `"/content/drive/MyDrive/.../base_model"`. |
| `ADAPTER_PATH` | Folder containing the saved LoRA adapter (it holds `adapter_config.json`), i.e. the folder the training notebook wrote with `model.save_pretrained(f"{OUT}/adapter")`. If left `""`, the **un-fine-tuned** base model runs. |
| `MOUNT_GOOGLE_DRIVE` | Set to `True` if the model or adapter lives in Google Drive. Cell 3 then mounts Drive. |

Nothing is pre-filled. If `MODEL_NAME_OR_PATH` is empty, Cell 3 stops with an explanatory error
instead of picking a model for you.

**Private or gated Hugging Face models:** store your token in Colab's **Secrets** panel (key icon in
the left sidebar) under the name `HF_TOKEN` and enable notebook access. Cell 3 reads it from there.
Never paste tokens into a cell.


In [ ]:
from __future__ import annotations

# ======================================================================================
#                                 EDIT THESE VALUES
# ======================================================================================

# --- Model (REQUIRED: nothing is assumed) -----------------------------------------------
MODEL_NAME_OR_PATH = ""      # <<< FILL IN: base model id or path (the one your adapter was trained on)
ADAPTER_PATH = ""            # <<< FILL IN if you use a LoRA adapter, e.g. "/content/drive/MyDrive/.../adapter"
TOKENIZER_NAME_OR_PATH = ""  # optional; leave empty to use the base model's tokenizer
MOUNT_GOOGLE_DRIVE = False   # True if the model/adapter is stored in Google Drive

# --- Device and precision -----------------------------------------------------------------
MODEL_DEVICE = "auto"        # "auto" (cuda if available), "cuda" or "cpu"
MODEL_DTYPE = "auto"         # "auto" (float16 on GPU, float32 on CPU), "float16", "bfloat16", "float32"
MERGE_ADAPTER = False        # True merges the LoRA weights into the base model (slightly faster inference)
TRUST_REMOTE_CODE = False    # only set True for models that require custom code you trust

# --- Generation (greedy decoding, as in the training notebook's evaluation) -------------------
MAX_NEW_TOKENS = 64          # the training notebook evaluated with 32; the answer is the first line
MAX_INPUT_TOKENS = 8192      # document + question tokens; the END of a longer document is cut
# Keep this BELOW ~100 s: Cloudflare tunnels cut requests off at about 100 s with an HTML error page.
GENERATION_TIMEOUT_SECONDS = 90

# --- Uploads -------------------------------------------------------------------------------
MAX_UPLOAD_MB = 20
MAX_PAGES = 100

# --- API server and CORS ---------------------------------------------------------------------
API_HOST = "127.0.0.1"       # keep as is: the tunnel connects locally; never expose 0.0.0.0
API_PORT = 8000
# Origins of the FRONTEND (where the browser page is served from), NOT the tunnel URL.
ALLOWED_ORIGINS = ["http://localhost:5173", "http://127.0.0.1:5173"]
START_TUNNEL = True          # False = start only the local server (no public URL)

# ======================================================================================
#                      END OF EDITABLE VALUES (settings class below)
# ======================================================================================


"""Application settings, read from environment variables (and an optional ``.env`` file).

Every value has a safe default except the model location: ``MODEL_NAME_OR_PATH``
and ``ADAPTER_PATH`` are intentionally empty because they are specific to your
training results and must be supplied by you. See ``.env.example``.
"""


from typing import Optional
from urllib.parse import urlparse

from pydantic import Field, field_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All tunable settings. Environment variable names are the upper-case field names."""

    # ``protected_namespaces=()`` allows field names that start with ``model_``.
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
        protected_namespaces=(),
    )

    # --- Model server (the app calls POST /predict on it) -----------------------
    # The fine-tuned model runs as its own local server (see the model repo's
    # scripts/start_server.sh). It builds the prompt itself; this app only sends the
    # extracted document text ("context") and the question.
    model_server_url: str = "http://127.0.0.1:8765"
    # Only needed if the model server was started with SERVING_API_KEY. Sent as X-API-Key.
    model_server_api_key: Optional[str] = None

    # --- In-process model (legacy; not used by the app any more) ---------------
    # Hugging Face model id or local directory of the BASE model used in training.
    model_name_or_path: Optional[str] = None
    # Directory containing the saved LoRA adapter (adapter_config.json, ...).
    adapter_path: Optional[str] = None
    # Optional tokenizer location; defaults to the base model's tokenizer.
    tokenizer_name_or_path: Optional[str] = None
    # auto | cpu | mps | cuda  ("auto" prefers cuda, then mps, then cpu)
    model_device: str = "auto"
    # auto | float32 | float16 | bfloat16
    model_dtype: str = "auto"
    # Merge the LoRA weights into the base model after loading (faster inference).
    merge_adapter: bool = False
    trust_remote_code: bool = False
    # Try to load the model when the server starts.
    load_model_on_startup: bool = True

    # --- Generation ------------------------------------------------------------
    # Decoding is always greedy (as in the training notebook's evaluation).
    # The notebook evaluated with 32; answers are short, so a small limit is enough.
    max_new_tokens: int = Field(64, ge=1, le=1024)
    # Upper bound on prompt tokens (document + question), also capped by the model's
    # context window. The whole document is sent; if it does not fit it is cut at the
    # END and the response says so. Larger values need more RAM / GPU memory.
    max_input_tokens: int = Field(8192, ge=256)
    generation_timeout_seconds: float = Field(120.0, gt=0)

    # --- Upload / extraction ---------------------------------------------------
    max_upload_mb: int = Field(20, ge=1, le=500)
    max_pages: int = Field(100, ge=1)
    # Minimum number of letters/digits that must be extracted for a PDF to count
    # as containing "meaningful text".
    min_text_chars: int = Field(20, ge=1)
    max_question_chars: int = Field(2000, ge=1)
    # Size limit of the Markdown text echoed back to the browser (the model still gets
    # the whole document, subject to MAX_INPUT_TOKENS).
    max_markdown_response_chars: int = Field(200_000, ge=1000)

    # --- API -------------------------------------------------------------------
    # Comma-separated list of allowed browser origins.
    cors_origins: str = "http://localhost:5173,http://127.0.0.1:5173"
    log_level: str = "INFO"

    @field_validator("model_server_url")
    @classmethod
    def _check_model_server_url(cls, value: str) -> str:
        value = value.strip().rstrip("/")
        parsed = urlparse(value)
        if parsed.scheme not in {"http", "https"} or not parsed.hostname:
            raise ValueError("MODEL_SERVER_URL must be an http(s) URL")
        # Plain HTTP is only acceptable on the local machine; anything else must use TLS.
        if parsed.scheme == "http" and parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
            raise ValueError("MODEL_SERVER_URL must use https unless it points to localhost")
        return value

    @field_validator(
        "model_name_or_path",
        "adapter_path",
        "tokenizer_name_or_path",
        "model_server_api_key",
        mode="before",
    )
    @classmethod
    def _blank_to_none(cls, value: object) -> object:
        if isinstance(value, str) and not value.strip():
            return None
        return value.strip() if isinstance(value, str) else value

    @field_validator("model_device")
    @classmethod
    def _check_device(cls, value: str) -> str:
        value = value.strip().lower()
        if value not in {"auto", "cpu", "mps", "cuda"}:
            raise ValueError("MODEL_DEVICE must be one of: auto, cpu, mps, cuda")
        return value

    @field_validator("model_dtype")
    @classmethod
    def _check_dtype(cls, value: str) -> str:
        value = value.strip().lower()
        if value not in {"auto", "float32", "float16", "bfloat16"}:
            raise ValueError(
                "MODEL_DTYPE must be one of: auto, float32, float16, bfloat16"
            )
        return value

    @property
    def max_upload_bytes(self) -> int:
        return self.max_upload_mb * 1024 * 1024

    @property
    def cors_origin_list(self) -> list[str]:
        return [o.strip() for o in self.cors_origins.split(",") if o.strip()]



settings = Settings(
    _env_file=None,  # ignore any .env file; the values above are the single source of truth
    model_name_or_path=MODEL_NAME_OR_PATH,
    adapter_path=ADAPTER_PATH,
    tokenizer_name_or_path=TOKENIZER_NAME_OR_PATH,
    model_device=MODEL_DEVICE,
    model_dtype=MODEL_DTYPE,
    merge_adapter=MERGE_ADAPTER,
    trust_remote_code=TRUST_REMOTE_CODE,
    max_new_tokens=MAX_NEW_TOKENS,
    max_input_tokens=MAX_INPUT_TOKENS,
    generation_timeout_seconds=GENERATION_TIMEOUT_SECONDS,
    max_upload_mb=MAX_UPLOAD_MB,
    max_pages=MAX_PAGES,
    cors_origins=",".join(ALLOWED_ORIGINS),
    load_model_on_startup=False,  # the model is loaded explicitly in Cell 3
)


def get_settings() -> Settings:
    """Used by create_app() (Cell 9) when no settings object is passed in."""
    return settings


print("Configuration OK")
print("  model   :", settings.model_name_or_path or "<<< NOT SET: fill in Cell 2")
print("  adapter :", settings.adapter_path or "(none: base model only)")
print("  limits  :", settings.max_input_tokens, "prompt tokens,", settings.max_new_tokens, "new tokens,", settings.max_pages, "pages")
print("  origins :", settings.cors_origin_list)


## Cell 3: Load the model

`ModelService` loads the tokenizer, the **base model**, and (if `ADAPTER_PATH` is set) the **LoRA adapter**
with `peft.PeftModel.from_pretrained`, then switches the model to evaluation (inference) mode.
It is loaded **once** and reused for every question.

* If `MOUNT_GOOGLE_DRIVE = True`, Drive is mounted first (Colab will ask you to authorise it).
* A Hugging Face token, if needed, is read from Colab Secrets (`HF_TOKEN`).
* Model loading is deliberately separate from the PDF extraction code below.
* It never substitutes another model: a missing or wrong path stops with an error.
* Like the training notebook, it sets `pad_token = eos_token` when the tokenizer has no pad token.

This class also contains the generation method (`generate_answer`) used by Cell 6 and the API.


In [ ]:
from __future__ import annotations

"""Model service: loads the configured SLM (+ optional LoRA adapter) once and answers questions.

Kept independent of PDF extraction. The rest of the app only depends on the small
``AnswerGenerator`` protocol below, so a remote inference implementation (e.g. an HTTP
client for a model hosted elsewhere) could replace ``ModelService`` without touching other
modules. No remote implementation exists.

Inference mirrors the training/evaluation notebook:

* the prompt is the plain-text training template (no chat template),
* decoding is greedy (``do_sample=False``),
* only newly generated tokens are decoded,
* the FIRST LINE of the generated text is the answer.

Heavy libraries (torch / transformers / peft) are imported lazily inside ``load`` so the
unit tests and the API can start without them being initialised.
"""


import logging
import os
import threading
import time
from dataclasses import dataclass
from typing import Any, Optional, Protocol


logger = logging.getLogger(__name__)


# --- Errors -------------------------------------------------------------------
class ModelServiceError(Exception):
    code = "model_error"

    def __init__(self, message: str) -> None:
        super().__init__(message)
        self.message = message


class ModelNotConfiguredError(ModelServiceError):
    code = "model_not_configured"


class ModelNotReadyError(ModelServiceError):
    code = "model_not_ready"


class ModelLoadError(ModelServiceError):
    code = "model_load_failed"


class InferenceError(ModelServiceError):
    code = "inference_failed"


class InferenceTimeoutError(ModelServiceError):
    code = "inference_timeout"


# --- Public result / interface --------------------------------------------------
@dataclass(frozen=True)
class GenerationResult:
    answer: str                 # first line of the generated text (may be "unanswerable")
    unanswerable: bool          # the model produced the training target "unanswerable"
    model_name: str
    prompt_tokens: Optional[int]      # None when the model server does not report it
    generated_tokens: Optional[int]   # None when the model server does not report it
    document_truncated: bool    # the document text had to be shortened to fit the model
    document_chars_total: Optional[int]
    document_chars_used: Optional[int]
    device: str
    dtype: str


class AnswerGenerator(Protocol):
    """Interface the API depends on."""

    @property
    def is_ready(self) -> bool: ...

    @property
    def model_name(self) -> str: ...

    @property
    def status(self) -> dict[str, Any]: ...

    def generate_answer(self, document_markdown: str, question: str) -> GenerationResult: ...


# --- Implementation -----------------------------------------------------------
class ModelService:
    """Local Transformers (+ PEFT) inference. The model is loaded once and reused."""

    def __init__(self, settings: Settings) -> None:
        self._settings = settings
        self._model: Any = None
        self._tokenizer: Any = None
        self._device = "unloaded"
        self._dtype = "unloaded"
        self._adapter_loaded = False
        self._load_error: Optional[str] = None
        self._load_lock = threading.Lock()
        self._generate_lock = threading.Lock()  # one generation at a time

    # -- state -------------------------------------------------------------------
    @property
    def is_ready(self) -> bool:
        return self._model is not None and self._tokenizer is not None

    @property
    def model_name(self) -> str:
        base = self._settings.model_name_or_path or "unconfigured"
        # Local paths are reduced to their last component to avoid leaking filesystem layout.
        if os.path.isabs(base) or base.startswith(("~", ".")):
            base = os.path.basename(os.path.normpath(os.path.expanduser(base)))
        if self._adapter_loaded and self._settings.adapter_path:
            adapter = os.path.basename(os.path.normpath(self._settings.adapter_path))
            return f"{base} + LoRA adapter '{adapter}'"
        return base

    @property
    def status(self) -> dict[str, Any]:
        return {
            "model_ready": self.is_ready,
            "model_name": self.model_name if self._settings.model_name_or_path else None,
            "adapter_loaded": self._adapter_loaded,
            "device": self._device,
            "load_error": self._load_error,
        }

    # -- loading -----------------------------------------------------------------
    def load(self) -> None:
        """Load tokenizer, base model and (optionally) the LoRA adapter.

        Raises ``ModelNotConfiguredError`` or ``ModelLoadError``. Safe to call again;
        it does nothing if the model is already loaded.
        """
        with self._load_lock:
            if self.is_ready:
                return
            s = self._settings
            if not s.model_name_or_path:
                msg = (
                    "MODEL_NAME_OR_PATH is not set. Set it (and ADAPTER_PATH if you use a "
                    "LoRA adapter) in the .env file."
                )
                self._load_error = msg
                raise ModelNotConfiguredError(msg)
            if s.adapter_path and not os.path.isdir(os.path.expanduser(s.adapter_path)):
                msg = "ADAPTER_PATH does not point to an existing directory."
                self._load_error = msg
                raise ModelLoadError(msg)

            try:
                self._load_unchecked()
                self._load_error = None
            except ModelServiceError as exc:
                self._load_error = exc.message
                raise
            except Exception as exc:
                logger.exception("Model loading failed")
                self._model = None
                self._tokenizer = None
                self._load_error = f"Model loading failed ({type(exc).__name__}). See server logs."
                raise ModelLoadError(self._load_error) from exc

    def _load_unchecked(self) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        s = self._settings
        device = self._resolve_device(torch)
        dtype_name, dtype = self._resolve_dtype(torch, device)
        logger.info("Loading model on device=%s dtype=%s", device, dtype_name)

        tokenizer = AutoTokenizer.from_pretrained(
            s.tokenizer_name_or_path or s.model_name_or_path,
            trust_remote_code=s.trust_remote_code,
        )
        model = AutoModelForCausalLM.from_pretrained(
            s.model_name_or_path,
            dtype=dtype,
            trust_remote_code=s.trust_remote_code,
        )

        adapter_loaded = False
        if s.adapter_path:
            from peft import PeftModel

            logger.info("Loading LoRA adapter")
            model = PeftModel.from_pretrained(model, os.path.expanduser(s.adapter_path))
            if s.merge_adapter:
                model = model.merge_and_unload()
            adapter_loaded = True

        model.to(device)
        model.eval()

        # Same as the training notebook.
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        self._tokenizer = tokenizer
        self._model = model
        self._device = device
        self._dtype = dtype_name
        self._adapter_loaded = adapter_loaded
        logger.info("Model ready")

    def _resolve_device(self, torch: Any) -> str:
        requested = self._settings.model_device
        if requested == "auto":
            if torch.cuda.is_available():
                return "cuda"
            if torch.backends.mps.is_available():
                return "mps"
            return "cpu"
        if requested == "cuda" and not torch.cuda.is_available():
            raise ModelLoadError("MODEL_DEVICE=cuda but CUDA is not available on this machine.")
        if requested == "mps" and not torch.backends.mps.is_available():
            raise ModelLoadError("MODEL_DEVICE=mps but Apple MPS is not available on this machine.")
        return requested

    def _resolve_dtype(self, torch: Any, device: str) -> tuple[str, Any]:
        requested = self._settings.model_dtype
        if requested == "auto":
            requested = "float32" if device == "cpu" else "float16"
        return requested, getattr(torch, requested)

    # -- inference -----------------------------------------------------------------
    def generate_answer(self, document_markdown: str, question: str) -> GenerationResult:
        """Build the training-format prompt, generate greedily and decode only new tokens."""
        if not self.is_ready:
            raise ModelNotReadyError(
                self._load_error or "The model is not loaded yet. Check /health for details."
            )
        import torch

        s = self._settings
        try:
            input_ids, used_chars = self._encode_within_budget(document_markdown, question)
        except ModelServiceError:
            raise
        except Exception as exc:
            logger.exception("Prompt construction failed")
            raise InferenceError("Failed to build the model prompt.") from exc
        prompt_tokens = int(input_ids.shape[-1])

        started = time.monotonic()
        try:
            with self._generate_lock, torch.inference_mode():
                output = self._model.generate(
                    input_ids=input_ids.to(self._device),
                    attention_mask=torch.ones_like(input_ids).to(self._device),
                    max_new_tokens=s.max_new_tokens,
                    max_time=s.generation_timeout_seconds,
                    do_sample=False,  # greedy, as in the notebook's evaluation
                    pad_token_id=self._tokenizer.pad_token_id,
                )
        except Exception as exc:
            logger.exception("Generation failed")
            raise InferenceError(f"Generation failed ({type(exc).__name__}).") from exc
        elapsed = time.monotonic() - started

        new_tokens = output[0][prompt_tokens:]
        generated_tokens = int(new_tokens.shape[-1])
        # max_time stops generation early; an unfinished, timed-out run is a timeout.
        hit_eos = generated_tokens > 0 and int(new_tokens[-1]) == self._tokenizer.eos_token_id
        if (
            elapsed >= s.generation_timeout_seconds
            and not hit_eos
            and generated_tokens < s.max_new_tokens
        ):
            raise InferenceTimeoutError(
                f"Answer generation exceeded {s.generation_timeout_seconds:.0f} seconds."
            )

        decoded = self._tokenizer.decode(new_tokens, skip_special_tokens=True)
        answer = first_line(decoded)
        if not answer:
            raise InferenceError("The model produced an empty answer.")

        return GenerationResult(
            answer=answer,
            unanswerable=is_unanswerable(answer),
            model_name=self.model_name,
            prompt_tokens=prompt_tokens,
            generated_tokens=generated_tokens,
            document_truncated=used_chars < len(document_markdown.strip()),
            document_chars_total=len(document_markdown.strip()),
            document_chars_used=used_chars,
            device=self._device,
            dtype=self._dtype,
        )

    def _token_budget(self) -> int:
        """Max prompt tokens: MAX_INPUT_TOKENS, capped by the model's context window."""
        s = self._settings
        limit = getattr(self._tokenizer, "model_max_length", None)
        if not isinstance(limit, int) or limit > 10_000_000:
            limit = getattr(getattr(self._model, "config", None), "max_position_embeddings", None)
        budget = s.max_input_tokens
        if isinstance(limit, int) and 0 < limit <= 10_000_000:
            budget = min(budget, limit - s.max_new_tokens)
        return budget

    def _encode(self, text: str) -> Any:
        # Same call as the training notebook: tok(prompt)["input_ids"] (default special tokens).
        return self._tokenizer(text, return_tensors="pt")["input_ids"]

    def _encode_within_budget(self, document: str, question: str) -> tuple[Any, int]:
        """Encode the full prompt; if too long, cut the END of the document (never the
        question or ``### Answer:``). Returns ``(input_ids, document_chars_used)``."""
        document = document.strip()
        budget = self._token_budget()
        ids = self._encode(build_prompt(document, question))
        if ids.shape[-1] <= budget:
            return ids, len(document)

        prefix, suffix = prompt_parts(question)

        def fits(n: int) -> tuple[bool, Any]:
            encoded = self._encode(prefix + document[:n].rstrip() + suffix)
            return encoded.shape[-1] <= budget, encoded

        if not fits(0)[0]:
            raise InferenceError(
                "The question is too long to fit the model's input limit. Shorten it."
            )
        low, high = 0, len(document)  # largest n with fits(n); fits(0) is True, fits(high) False
        while high - low > 1:
            mid = (low + high) // 2
            if fits(mid)[0]:
                low = mid
            else:
                high = mid
        # Avoid cutting inside a word: back up to the previous whitespace when possible.
        cut = low
        boundary = max(document.rfind(" ", 0, cut), document.rfind("\n", 0, cut))
        if boundary > cut * 0.9:
            cut = boundary
        ok, encoded = fits(cut)
        if not ok:  # extremely unlikely; fall back to the exact fitting length
            cut = low
            _, encoded = fits(cut)
        logger.info("Document truncated to %d of %d characters", cut, len(document))
        return encoded, len(document[:cut].rstrip())




# --------------------------------------------------------------------------------------
# Load the configured model (runs once)
# --------------------------------------------------------------------------------------
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

# A Hugging Face token is only needed for gated/private models. Read it from Colab Secrets.
try:
    from google.colab import userdata

    _hf_token = userdata.get("HF_TOKEN")
    if _hf_token:
        os.environ["HF_TOKEN"] = _hf_token
except Exception:  # not running on Colab, or no HF_TOKEN secret / access not granted
    pass

model_service = ModelService(settings)
try:
    model_service.load()
except ModelServiceError as exc:
    print("MODEL NOT LOADED:", exc.message)
    raise
print("Model ready:", model_service.status)


## Cell 4: Structure-preserving PDF extraction

`process_pdf(filename, data, max_bytes=...)` validates the upload and reads the PDF with PyMuPDF
**keeping its structure**, then `StructuredDocument.markdown` renders it as Markdown:

| PDF content | Markdown |
|---|---|
| title / headings | `#`, `##`, `###`, `####` (font size relative to the body text, bold, section numbers such as `1.1`) |
| ruled tables | `\| col \| col \|` rows with a `\| --- \|` separator |
| bullets / numbered items | `- item` / `1. item` |
| body text | paragraphs separated by blank lines |

Page numbers, repeated headers and footers are removed. Every element remembers its page so the answer
can later be located.

It raises a specific error for a wrong file type (extension **and** `%PDF-` signature), an empty file,
a file over `MAX_UPLOAD_MB`, too many pages, a corrupt or password-protected PDF, and a PDF with **no
meaningful text**. **No OCR:** scanned PDFs are rejected, not guessed at.

These are heuristics: unusual layouts (multi-column pages, tables without ruling lines, headings styled
like body text) may be read imperfectly.


In [ ]:
from __future__ import annotations

"""PDF validation and structure-preserving text extraction (text-based PDFs only).

``process_pdf`` validates an upload and returns a ``StructuredDocument``: an ordered list
of headings, paragraphs, list items and tables, each tagged with its page number. The
structure is recovered with simple, explainable heuristics on top of PyMuPDF:

* **Body font size** = the most common font size (weighted by characters).
* **Headings** = short lines that are noticeably larger than the body text
  (>= 1.5x title, >= 1.25x heading 1, >= 1.10x heading 2) or fully bold
  (heading 3). Section numbers such as ``1.``, ``1.1``, ``1.1.1`` give the level.
* **Lists** = lines that start with a bullet glyph or a number such as ``1.`` / ``(a)``.
* **Tables** = ruled tables found by ``page.find_tables()``; text inside a table's
  bounding box is not repeated as paragraphs.
* **Paragraphs** = consecutive lines with small vertical gaps.
* Repeated page headers/footers and bare page numbers are dropped.

Limitations: scanned/image-only PDFs are NOT supported (no OCR); tables without ruling
lines are read as ordinary text; multi-column layouts may be read in the wrong order.
"""


import logging
import os
import re
from collections import Counter
from dataclasses import dataclass
from typing import Any, Optional

import pymupdf


logger = logging.getLogger(__name__)


# --- Errors -----------------------------------------------------------------
class DocumentProcessingError(Exception):
    """Base class for PDF processing problems. ``code`` is machine readable."""

    code = "document_error"

    def __init__(self, message: str) -> None:
        super().__init__(message)
        self.message = message


class UnsupportedFileTypeError(DocumentProcessingError):
    code = "unsupported_file_type"


class EmptyFileError(DocumentProcessingError):
    code = "empty_file"


class FileTooLargeError(DocumentProcessingError):
    code = "file_too_large"


class TooManyPagesError(DocumentProcessingError):
    code = "too_many_pages"


class MalformedPDFError(DocumentProcessingError):
    code = "malformed_pdf"


class NoExtractableTextError(DocumentProcessingError):
    code = "no_extractable_text"


# --- Validation helpers -------------------------------------------------------
_PDF_MAGIC = b"%PDF-"
_ALNUM = re.compile(r"[^\W_]", re.UNICODE)  # letters and digits


def sanitize_filename(filename: str | None) -> str:
    """Return a display-safe filename. It is only a label and is never used as a path."""
    name = os.path.basename((filename or "").replace("\\", "/")).strip()
    name = re.sub(r"[\x00-\x1f\x7f]", "", name)
    return name[:255] or "document.pdf"


def validate_pdf_upload(filename: str | None, data: bytes, max_bytes: int) -> None:
    """Validate extension, size and PDF signature. Raises ``DocumentProcessingError``.

    The client-supplied Content-Type header is deliberately ignored because it is
    trivially spoofed; the file signature is checked instead.
    """
    if not filename or not filename.lower().endswith(".pdf"):
        raise UnsupportedFileTypeError("Only files with a .pdf extension are supported.")
    if len(data) == 0:
        raise EmptyFileError("The uploaded file is empty.")
    if len(data) > max_bytes:
        raise FileTooLargeError(
            f"The uploaded file exceeds the maximum size of {max_bytes // (1024 * 1024)} MB."
        )
    if _PDF_MAGIC not in data[:1024]:
        raise UnsupportedFileTypeError("The uploaded file is not a valid PDF document.")


# --- Heuristic thresholds (relative to the body font size) ----------------------
TITLE_RATIO = 1.5
H1_RATIO = 1.25
H2_RATIO = 1.10
BOLD_HEADING_MIN_RATIO = 0.95
MAX_HEADING_CHARS = 120
PARAGRAPH_GAP_FACTOR = 0.5  # vertical gap (x font size) that starts a new paragraph
MARGIN_FRACTION = 0.08      # top/bottom page fraction searched for headers/footers

_SECTION_NUMBER = re.compile(r"^(\d+(?:\.\d+)*)[\.\)]?\s+\S")
_PAGE_NUMBER_ONLY = re.compile(r"^(?:page\s*)?\d+(?:\s*(?:of|/)\s*\d+)?$", re.IGNORECASE)


@dataclass
class _Line:
    text: str
    size: float
    bold: bool
    x0: float
    y0: float
    y1: float


# --- Low-level page reading -----------------------------------------------------
def _span_is_bold(span: dict[str, Any]) -> bool:
    font = str(span.get("font", "")).lower()
    return bool(span.get("flags", 0) & 16) or any(w in font for w in ("bold", "black", "heavy"))


def _inside(bbox: tuple[float, float, float, float], x: float, y: float, pad: float = 2.0) -> bool:
    return bbox[0] - pad <= x <= bbox[2] + pad and bbox[1] - pad <= y <= bbox[3] + pad


def _clean_cell(value: Any) -> str:
    return " ".join(str(value).split()) if value is not None else ""


def _extract_tables(page: Any) -> list[tuple[tuple[float, float, float, float], list[list[str]]]]:
    """Return ``[(bbox, rows)]`` for ruled tables on the page (empty on any failure)."""
    found: list[tuple[tuple[float, float, float, float], list[list[str]]]] = []
    try:
        finder = page.find_tables()
        for table in finder.tables:
            rows = [[_clean_cell(c) for c in row] for row in table.extract()]
            rows = [row for row in rows if any(row)]  # drop empty rows
            if not rows:
                continue
            width = max(len(r) for r in rows)
            rows = [r + [""] * (width - len(r)) for r in rows]
            keep = [i for i in range(width) if any(r[i] for r in rows)]  # drop empty columns
            rows = [[r[i] for i in keep] for r in rows]
            if len(rows) >= 2 and rows and len(rows[0]) >= 2:  # ignore 1-row/1-column "tables"
                found.append((tuple(table.bbox), rows))
    except Exception as exc:  # table detection is best-effort
        logger.warning("Table detection failed on a page: %s", type(exc).__name__)
    return found


def _read_lines(page: Any, table_boxes: list[tuple[float, float, float, float]]) -> list[_Line]:
    """Text lines in reading order, excluding lines that lie inside detected tables."""
    lines: list[_Line] = []
    page_dict = page.get_text("dict", sort=True)
    for block in page_dict.get("blocks", []):
        if block.get("type") != 0:  # 0 = text block (1 = image)
            continue
        for line in block.get("lines", []):
            spans = [s for s in line.get("spans", []) if str(s.get("text", "")).strip()]
            if not spans:
                continue
            x0, y0, x1, y1 = line["bbox"]
            if any(_inside(box, (x0 + x1) / 2, (y0 + y1) / 2) for box in table_boxes):
                continue
            text = " ".join("".join(str(s["text"]) for s in line["spans"]).split())
            text = text.replace("\x00", "")
            if not text:
                continue
            dominant = max(spans, key=lambda s: len(str(s["text"]).strip()))
            lines.append(
                _Line(
                    text=text,
                    size=round(float(dominant.get("size", 0.0)) * 2) / 2,  # nearest 0.5pt
                    bold=all(_span_is_bold(s) for s in spans),
                    x0=float(x0),
                    y0=float(y0),
                    y1=float(y1),
                )
            )
    return lines


def _drop_headers_footers(
    pages_lines: list[tuple[list[_Line], float]],
) -> list[list[_Line]]:
    """Remove bare page numbers and (for >=3 pages) text repeated in the page margins."""
    page_count = len(pages_lines)
    seen: Counter[str] = Counter()

    def in_margin(line: _Line, height: float) -> bool:
        return line.y1 <= height * MARGIN_FRACTION or line.y0 >= height * (1 - MARGIN_FRACTION)

    def signature(text: str) -> str:
        return re.sub(r"\d+", "#", text.lower())

    for lines, height in pages_lines:
        for sig in {signature(l.text) for l in lines if in_margin(l, height)}:
            seen[sig] += 1

    repeated_min = max(2, (page_count + 1) // 2)
    cleaned: list[list[_Line]] = []
    for lines, height in pages_lines:
        kept = []
        for line in lines:
            if in_margin(line, height):
                if _PAGE_NUMBER_ONLY.match(line.text):
                    continue
                if page_count >= 3 and seen[signature(line.text)] >= repeated_min:
                    continue
            kept.append(line)
        cleaned.append(kept)
    return cleaned


# --- Classification and assembly --------------------------------------------------
def _body_font_size(all_lines: list[_Line]) -> float:
    weights: Counter[float] = Counter()
    for line in all_lines:
        weights[line.size] += len(line.text)
    return weights.most_common(1)[0][0] if weights else 0.0


def _heading_type(line: _Line, body: float, first_title_allowed: bool) -> Optional[str]:
    """Return heading_1/2/3 or title for a heading-like line, otherwise ``None``."""
    text = line.text
    if len(text) > MAX_HEADING_CHARS:
        return None
    number = _SECTION_NUMBER.match(text)
    if text.endswith((",", ";")) or (text.endswith(".") and not number):
        return None
    ratio = line.size / body if body else 1.0

    size_based: Optional[str] = None
    if ratio >= TITLE_RATIO:
        size_based = "title" if first_title_allowed else "heading_1"
    elif ratio >= H1_RATIO:
        size_based = "heading_1"
    elif ratio >= H2_RATIO:
        size_based = "heading_2"

    bold_based = line.bold and ratio >= BOLD_HEADING_MIN_RATIO and len(text) <= 100
    if size_based is None and not bold_based:
        return None
    if number and (size_based or bold_based) and size_based != "title":
        depth = number.group(1).count(".") + 1
        return f"heading_{min(depth, 3)}"
    return size_based or "heading_3"


def _is_list_start(text: str) -> bool:
    return bool(BULLET_MARKER.match(text) or NUMBERED_MARKER.match(text))


def _join(previous: str, nxt: str) -> str:
    """Join two lines of the same paragraph, repairing end-of-line hyphenation."""
    if previous.endswith("-") and len(previous) > 1 and previous[-2].isalpha() and nxt[:1].islower():
        return previous[:-1] + nxt
    return f"{previous} {nxt}"


def _build_text_elements(
    lines: list[_Line], page_number: int, body: float, state: dict[str, bool]
) -> list[tuple[float, DocumentElement]]:
    """Group classified lines into elements. Returns ``(y0, element)`` in reading order."""
    out: list[tuple[float, DocumentElement]] = []
    current_type: Optional[str] = None
    current_text = ""
    current_y0 = 0.0
    previous: Optional[_Line] = None

    def flush() -> None:
        nonlocal current_type, current_text
        if current_type and current_text.strip():
            out.append(
                (current_y0, DocumentElement(current_type, current_text.strip(), page_number))
            )
        current_type, current_text = None, ""

    for line in lines:
        heading = _heading_type(line, body, first_title_allowed=not state["title_seen"])
        if heading == "title":
            state["title_seen"] = True
        if heading:
            kind = heading
        elif _is_list_start(line.text):
            kind = "list_item"
        else:
            kind = "paragraph"

        gap = (line.y0 - previous.y1) if previous else 0.0
        tight = previous is not None and gap < PARAGRAPH_GAP_FACTOR * max(line.size, 1.0)
        starts_numbered_heading = bool(heading and _SECTION_NUMBER.match(line.text))

        continues = False
        if current_type is not None and tight:
            if kind == "paragraph" and current_type in ("paragraph", "list_item"):
                continues = True  # wrapped paragraph line / list-item continuation
            elif kind == current_type and heading and not starts_numbered_heading:
                continues = line.size == (previous.size if previous else line.size)

        if continues:
            current_text = _join(current_text, line.text)
        else:
            flush()
            current_type, current_text, current_y0 = kind, line.text, line.y0
        previous = line
    flush()
    return out


def _merge_tables(
    text_items: list[tuple[float, DocumentElement]],
    tables: list[tuple[tuple[float, float, float, float], list[list[str]]]],
    page_number: int,
) -> list[DocumentElement]:
    """Place each table before the first text element that lies below its top edge."""
    pending = sorted(tables, key=lambda t: t[0][1])
    merged: list[DocumentElement] = []
    for y0, element in text_items:
        while pending and pending[0][0][1] <= y0:
            merged.append(DocumentElement("table", pending.pop(0)[1], page_number))
        merged.append(element)
    for _, rows in pending:
        merged.append(DocumentElement("table", rows, page_number))
    return merged


# --- Public entry point ------------------------------------------------------------
def process_pdf(
    filename: str | None,
    data: bytes,
    *,
    max_bytes: int,
    min_text_chars: int = 20,
    max_pages: int = 100,
) -> StructuredDocument:
    """Validate a PDF and extract its structure.

    Raises:
        UnsupportedFileTypeError, EmptyFileError, FileTooLargeError: invalid upload.
        TooManyPagesError: more than ``max_pages`` pages.
        MalformedPDFError: corrupt, encrypted or otherwise unreadable PDF.
        NoExtractableTextError: no meaningful text (e.g. scanned PDF without OCR).
    """
    validate_pdf_upload(filename, data, max_bytes)

    try:
        doc = pymupdf.open(stream=data, filetype="pdf")
    except Exception as exc:  # PyMuPDF raises several exception types
        logger.warning("PDF could not be opened: %s", type(exc).__name__)
        raise MalformedPDFError("The PDF is corrupt or could not be read.") from exc

    try:
        if doc.needs_pass:
            raise MalformedPDFError("Password-protected PDFs are not supported.")
        page_count = doc.page_count
        if page_count == 0:
            raise MalformedPDFError("The PDF contains no pages.")
        if page_count > max_pages:
            raise TooManyPagesError(
                f"The PDF has {page_count} pages; the maximum supported is {max_pages}."
            )

        page_tables: list[list[tuple[tuple[float, float, float, float], list[list[str]]]]] = []
        page_lines: list[tuple[list[_Line], float]] = []
        for index in range(page_count):
            try:
                page = doc.load_page(index)
                tables = _extract_tables(page)
                lines = _read_lines(page, [bbox for bbox, _ in tables])
                height = float(page.rect.height)
            except DocumentProcessingError:
                raise
            except Exception as exc:
                logger.warning(
                    "Extraction failed on page %d: %s", index + 1, type(exc).__name__
                )
                raise MalformedPDFError(f"Text extraction failed on page {index + 1}.") from exc
            page_tables.append(tables)
            page_lines.append((lines, height))
    finally:
        doc.close()

    cleaned = _drop_headers_footers(page_lines)
    body = _body_font_size([line for lines in cleaned for line in lines])

    elements: list[DocumentElement] = []
    empty_pages: list[int] = []
    state = {"title_seen": False}
    for index, lines in enumerate(cleaned):
        page_number = index + 1
        items = _build_text_elements(lines, page_number, body, state)
        page_elements = _merge_tables(items, page_tables[index], page_number)
        if not page_elements:
            empty_pages.append(page_number)
        elements.extend(page_elements)

    def element_text(e: DocumentElement) -> str:
        return " ".join(c for row in e.content for c in row) if e.type == "table" else str(e.content)

    meaningful = sum(len(_ALNUM.findall(element_text(e))) for e in elements)
    if meaningful < min_text_chars:
        raise NoExtractableTextError(
            "No meaningful text could be extracted. The PDF may be scanned or "
            "image-only; OCR is not supported."
        )

    return StructuredDocument(
        elements=elements, page_count=page_count, empty_page_numbers=empty_pages
    )


## Cell 5: Prompt

`build_prompt(context, question)` produces the text the model receives:

```
You are a senior business and finance analyst at a reputed organization. Your task is to analyze
the provided document content and answer the user's question accurately and concisely.

**Document Content:**
{document as Markdown}

**Question:**
{question}

**Instructions:**
- Answer the question using only the information provided in the document.
- Provide a precise, crisp, and professional answer that directly addresses the question.
- Avoid unnecessary explanations, background information, or repetition.
- Do not make assumptions or introduce information that is not supported by the document.
- If the answer is not available in the provided content, respond with: "The requested information is not available in the provided document."

**Answer:**
```

No chat template is added. The model continues after `**Answer:**` and its **first line** is the
answer. The sentence "The requested information is not available in the provided document." (or the
bare word `unanswerable`) is reported as an unanswerable result.

`structure.py` (included in this cell) also holds `locate_answer`: after generation it **searches the
extracted document for the answer text** and reports the page, the heading path and, for tables, the
row and column labels. This is plain string matching, **not model output**; it returns nothing when the
answer text does not appear verbatim.


In [ ]:
from __future__ import annotations

"""Structured document model: headings, paragraphs, lists and tables -> Markdown.

The extractor (``document_processor.py``) produces a list of ``DocumentElement``. This module
turns that list into the Markdown text that is placed in the model prompt:

* title / headings -> ``#`` / ``##`` / ``###`` / ``####`` so the model sees the hierarchy
* tables           -> Markdown tables (``| a | b |`` rows with a ``| --- | --- |`` separator)
* list items       -> ``- item`` lines
* paragraphs       -> plain text separated by blank lines

It also contains ``locate_answer``: a plain string match of the model's answer against the
extracted elements, used to tell the user *where* in the document the answer text appears.
This is NOT produced by the model and is only reported when an actual match is found.
"""


import re
from dataclasses import dataclass, field
from typing import Any, Optional

# element.type -> Markdown heading marker
HEADING_MARKERS = {"title": "#", "heading_1": "##", "heading_2": "###", "heading_3": "####"}
# element.type -> nesting depth used to build the "section path"
HEADING_LEVELS = {"title": 0, "heading_1": 1, "heading_2": 2, "heading_3": 3}

# Bullet glyphs (including U+00B7 and the Symbol-font private-use bullets U+F0B7/U+F0A7) may be
# followed directly by the text; ASCII/dash markers need a following space.
BULLET_MARKER = re.compile(r"^(?:[•▪◦‣●○■□·\uf0b7\uf0a7]\s*|[\-\*–—]\s+)")
NUMBERED_MARKER = re.compile(r"^(?:\d+[\.\)]|\([a-zA-Z0-9]+\))\s+")
LIST_MARKER = re.compile(f"(?:{BULLET_MARKER.pattern})|(?:{NUMBERED_MARKER.pattern})")


@dataclass(frozen=True)
class DocumentElement:
    """One structural element in reading order.

    ``content`` is a ``str`` for text elements and a ``list[list[str]]`` (rows of cells,
    first row = header) for tables. ``page_number`` is 1-based.
    """

    type: str  # title | heading_1 | heading_2 | heading_3 | paragraph | list_item | table
    content: Any
    page_number: int


@dataclass
class StructuredDocument:
    elements: list[DocumentElement]
    page_count: int
    empty_page_numbers: list[int] = field(default_factory=list)

    @property
    def markdown(self) -> str:
        return elements_to_markdown(self.elements)

    @property
    def heading_count(self) -> int:
        return sum(1 for e in self.elements if e.type in HEADING_MARKERS)

    @property
    def table_count(self) -> int:
        return sum(1 for e in self.elements if e.type == "table")


# --------------------------------------------------------------------------------------
# Markdown rendering
# --------------------------------------------------------------------------------------
def _escape_cell(cell: str) -> str:
    return cell.replace("|", "\\|")


def table_to_markdown(grid: list[list[str]]) -> str:
    """Render rows of cells as a Markdown table (first row is the header)."""
    if not grid:
        return ""
    columns = max(len(row) for row in grid)
    rows = [[_escape_cell(c) for c in row] + [""] * (columns - len(row)) for row in grid]
    lines = ["| " + " | ".join(rows[0]) + " |", "| " + " | ".join(["---"] * columns) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in rows[1:]]
    return "\n".join(lines)


def element_to_markdown(element: DocumentElement) -> str:
    if element.type in HEADING_MARKERS:
        return f"{HEADING_MARKERS[element.type]} {element.content}"
    if element.type == "list_item":
        text = str(element.content).strip()
        if NUMBERED_MARKER.match(text):
            return text  # keep "1. item" / "(a) item" as written
        return "- " + BULLET_MARKER.sub("", text, count=1)
    if element.type == "table":
        return table_to_markdown(element.content)
    return str(element.content)


def elements_to_markdown(elements: list[DocumentElement]) -> str:
    """Join elements with blank lines; consecutive list items stay on adjacent lines."""
    parts: list[str] = []
    previous: Optional[str] = None
    for element in elements:
        rendered = element_to_markdown(element)
        if not rendered:
            continue
        if parts and previous == "list_item" and element.type == "list_item":
            parts[-1] += "\n" + rendered
        else:
            parts.append(rendered)
        previous = element.type
    return "\n\n".join(parts).strip()


# --------------------------------------------------------------------------------------
# Answer location (string matching, not model output)
# --------------------------------------------------------------------------------------
@dataclass(frozen=True)
class TableCellRef:
    value: str
    row_label: Optional[str]     # first-column value of the row (None for header/first column)
    column_label: Optional[str]  # header text of the column (None when the match is in the header)


@dataclass(frozen=True)
class AnswerMatch:
    page_number: int
    element_type: str
    section_path: tuple[str, ...]  # headings above the match, outermost first
    snippet: str
    table_cell: Optional[TableCellRef] = None


_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)


def normalize(text: str) -> str:
    """Lower-case, replace punctuation by spaces, collapse whitespace."""
    return " ".join(_PUNCT.sub(" ", text.lower()).split())


def _contains_phrase(haystack: str, needle: str) -> bool:
    """Whole-word phrase match on normalized text (so '2' does not match '21')."""
    return f" {needle} " in f" {haystack} "


def _snippet(text: str, answer: str, width: int = 300) -> str:
    text = " ".join(text.split())
    index = text.lower().find(answer.lower().strip(" .")) if answer.strip(" .") else -1
    start = max(0, index - width // 3) if index > 0 else 0
    piece = text[start : start + width]
    return ("..." if start > 0 else "") + piece + ("..." if start + width < len(text) else "")


def locate_answer(
    elements: list[DocumentElement], answer: str, max_results: int = 5
) -> list[AnswerMatch]:
    """Find where the answer text occurs in the extracted document.

    Returns up to ``max_results`` locations in reading order, each with the page, the heading
    path (section) and, for tables, the row/column labels. Returns ``[]`` when the answer text
    does not occur verbatim (after normalisation) in any element.
    """
    needle = normalize(answer)
    if not needle:
        return []

    results: list[AnswerMatch] = []
    path: list[tuple[int, str]] = []  # (heading level, heading text)

    for element in elements:
        if element.type in HEADING_LEVELS:
            level = HEADING_LEVELS[element.type]
            while path and path[-1][0] >= level:
                path.pop()
            path.append((level, str(element.content)))

        if len(results) >= max_results:
            break
        section = tuple(text for _, text in path)

        if element.type == "table":
            grid: list[list[str]] = element.content
            for r, row in enumerate(grid):
                for c, cell in enumerate(row):
                    if len(results) < max_results and _contains_phrase(normalize(cell), needle):
                        ref = TableCellRef(
                            value=cell,
                            row_label=row[0] if r > 0 and c > 0 else None,
                            column_label=grid[0][c] if r > 0 and c < len(grid[0]) else None,
                        )
                        results.append(
                            AnswerMatch(
                                page_number=element.page_number,
                                element_type="table",
                                section_path=section,
                                snippet=_snippet(" | ".join(row), answer),
                                table_cell=ref,
                            )
                        )
        elif _contains_phrase(normalize(str(element.content)), needle):
            results.append(
                AnswerMatch(
                    page_number=element.page_number,
                    element_type=element.type,
                    section_path=section,
                    snippet=_snippet(str(element.content), answer),
                )
            )
    return results


"""Prompt construction.

The prompt is the instruction template supplied by the project owner::

    You are a senior business and finance analyst ... answer the user's question accurately and concisely.

    **Document Content:**
    {content}

    **Question:**
    {question}

    **Instructions:**
    - ...

    **Answer:**

``{content}`` is the whole document as structure-preserving Markdown (headings as
``#``/``##``, tables as ``| a | b |`` rows, list items as ``- item``). The text ends right after
``**Answer:**`` and a newline; the model continues from there.
"""


NOT_AVAILABLE_ANSWER = "The requested information is not available in the provided document."

_INTRO = (
    "You are a senior business and finance analyst at a reputed organization. "
    "Your task is to analyze the provided document content and answer the user's question "
    "accurately and concisely.\n\n"
)
CONTENT_HEADER = _INTRO + "**Document Content:**\n"
QUESTION_HEADER = "\n\n**Question:**\n"
INSTRUCTIONS_AND_ANSWER_HEADER = (
    "\n\n**Instructions:**\n"
    "- Answer the question using only the information provided in the document.\n"
    "- Provide a precise, crisp, and professional answer that directly addresses the question.\n"
    "- Avoid unnecessary explanations, background information, or repetition.\n"
    "- Do not make assumptions or introduce information that is not supported by the document.\n"
    f'- If the answer is not available in the provided content, respond with: "{NOT_AVAILABLE_ANSWER}"\n'
    "\n**Answer:**\n"
)


def build_prompt(context: str, question: str) -> str:
    """Return the full prompt (ending right after ``**Answer:**\\n``)."""
    prefix, suffix = prompt_parts(question)
    return f"{prefix}{context.strip()}{suffix}"


def prompt_parts(question: str) -> tuple[str, str]:
    """Return ``(prefix, suffix)`` so that ``prefix + context + suffix == build_prompt(...)``.

    The model service uses this to shorten only the document part when the prompt would
    exceed the token budget, keeping the question, the instructions and ``**Answer:**`` intact.
    """
    return CONTENT_HEADER, f"{QUESTION_HEADER}{question.strip()}{INSTRUCTIONS_AND_ANSWER_HEADER}"


def first_line(generated: str) -> str:
    """Use the first line of the generated text as the answer."""
    stripped = generated.strip()
    return stripped.splitlines()[0].strip() if stripped else ""


def is_unanswerable(answer: str) -> bool:
    """True when the model says the document does not contain the answer.

    Matches the sentence the prompt asks for, and the bare label ``unanswerable``.
    """
    normalized = answer.strip().strip('"').strip().rstrip(".").lower()
    return normalized in {"unanswerable", NOT_AVAILABLE_ANSWER.rstrip(".").lower()}


## Cell 6: Model inference (smoke test)

`model_service.generate_answer(document_markdown, question)` builds the prompt, runs **greedy**
generation and decodes **only the newly generated tokens**. If the prompt would exceed
`MAX_INPUT_TOKENS` (or the model's context window), the END of the document is cut; the question and
`**Answer:**` are never cut, and the result says `document_truncated`.

This cell calls the **real loaded model** once on a tiny test document so you can confirm that
inference works before starting the API. The document below is test input only; it is not used by the
application. Judge the answer yourself: this is a functional check, not an evaluation.


In [ ]:
import time

TEST_DOCUMENT = (
    "# Meridian Report\n\n## 1. Summary\n\nThe Meridian Report was published in 2021. "
    "Its main recommendation is that regional libraries extend their weekend opening hours.\n\n"
    "| Library | Opening hours |\n| --- | --- |\n| North | 9-17 |\n| South | 10-16 |"
)
TEST_QUESTION = "What is the main recommendation of the Meridian Report?"

_started = time.monotonic()
try:
    _result = model_service.generate_answer(TEST_DOCUMENT, TEST_QUESTION)
except ModelServiceError as exc:
    print("INFERENCE FAILED:", exc.message)
    raise
_elapsed = time.monotonic() - _started

print("Question :", TEST_QUESTION)
print("Answer   :", _result.answer, "(unanswerable)" if _result.unanswerable else "")
print(f"Model    : {_result.model_name}")
print(f"Time     : {_elapsed:.2f} s  (prompt tokens: {_result.prompt_tokens}, generated tokens: {_result.generated_tokens})")


## Cell 7: FastAPI application and CORS

`create_app()` builds the API:

| Method | Path | Purpose |
|---|---|---|
| GET | `/health` | server status and real model readiness |
| POST | `/answer` | multipart form with `file` (PDF) and `question`: extract, prompt, run the SLM, return the answer |

The response contains the answer, the model name, the measured inference time, the extraction facts
(pages, headings, tables, words), the structured Markdown that was given to the model, and
`answer_locations` (string matches of the answer in the document). All errors share one JSON shape,
`{"error": {"code": "...", "message": "..."}}`, and never include stack traces or file paths.
Document contents are not logged and nothing is stored.

**CORS.** A browser only lets the page at `http://localhost:5173` call this API if the API names that
origin. `ALLOWED_ORIGINS` (Cell 2) is that list; `create_app()` passes it to `CORSMiddleware`
(methods GET, POST, OPTIONS; no credentials). The browser's `Origin` is the *frontend's* origin, not
the tunnel URL, so the tunnel URL does **not** go into `ALLOWED_ORIGINS`. A wildcard `*` is refused below,
and the API is only ever bound to localhost.


In [ ]:
from __future__ import annotations

"""Shared API error type.

Every error returned by the API has the same JSON shape::

    {"error": {"code": "<machine_readable_code>", "message": "<human readable>"}}
"""


class ApiError(Exception):
    """An error that should be returned to the client with a specific HTTP status."""

    def __init__(self, status_code: int, code: str, message: str) -> None:
        super().__init__(message)
        self.status_code = status_code
        self.code = code
        self.message = message


"""Pydantic response models (the public API contract)."""


from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class ErrorDetail(BaseModel):
    code: str
    message: str


class ErrorResponse(BaseModel):
    error: ErrorDetail


class HealthResponse(BaseModel):
    model_config = ConfigDict(protected_namespaces=())

    status: str = Field(examples=["ok"])
    model_ready: bool
    model_name: Optional[str] = None
    adapter_loaded: bool = False
    device: Optional[str] = None
    model_error: Optional[str] = None


class DocumentInfo(BaseModel):
    """Facts measured while extracting the uploaded PDF."""

    filename: str
    page_count: int
    heading_count: int
    table_count: int
    character_count: int = Field(description="Characters in the structured (Markdown) text.")
    word_count: int
    empty_page_numbers: list[int] = Field(
        description="1-based pages from which no text could be extracted."
    )


class TableCell(BaseModel):
    value: str
    row_label: Optional[str] = Field(None, description="First-column value of the matching row.")
    column_label: Optional[str] = Field(None, description="Header text of the matching column.")


class AnswerLocation(BaseModel):
    """Where the answer text literally occurs in the extracted document.

    Found by plain string matching of the model's answer against the extracted elements.
    It is NOT produced by the model and is not a citation of the model's reasoning.
    """

    page_number: int
    element_type: str = Field(
        description="paragraph | list_item | table | title | heading_1 | heading_2 | heading_3"
    )
    section_path: list[str] = Field(description="Headings above the match, outermost first.")
    snippet: str
    table_cell: Optional[TableCell] = None


class AnswerResponse(BaseModel):
    model_config = ConfigDict(protected_namespaces=())

    question: str
    answer: str = Field(description="First line of the text generated by the model.")
    unanswerable: bool = Field(
        description="True when the model produced its training label 'unanswerable', i.e. it "
        "judged that the document does not contain the answer."
    )
    message: Optional[str] = None
    model_name: str
    inference_time_seconds: float
    prompt_tokens: Optional[int] = Field(None, description="Not reported by the model server.")
    generated_tokens: Optional[int] = Field(None, description="Not reported by the model server.")

    document: DocumentInfo
    document_truncated: bool = Field(
        description="True when the document did not fit the model's input limit and its "
        "END was cut. The model then answered from the beginning only."
    )
    document_chars_used: Optional[int] = Field(None, description="Not reported by the model server.")
    document_chars_total: Optional[int] = Field(None, description="Not reported by the model server.")

    structured_text: str = Field(description="The Markdown given to the model (bounded).")
    structured_text_truncated: bool

    answer_locations: list[AnswerLocation]
    disclaimer: str = (
        "Answers are generated by a small language model and may be incorrect. "
        "Check them against the document."
    )


"""FastAPI application: upload a PDF and a question in ONE request, get an answer.

Flow of ``POST /answer``::

    PDF + question --> structure-preserving extraction (Markdown)
                   --> POST {context, question} to the model server's /predict
                       (the model server builds the prompt and runs the fine-tuned SLM)
                   --> answer

There is no retrieval, no chunking and no server-side storage: nothing is kept after the
response is sent.

Run with:  ``uvicorn app.main:app --host 127.0.0.1 --port 8000``
"""


import logging
import time
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI, File, Form, Request, UploadFile
from fastapi.concurrency import run_in_threadpool
from fastapi.exceptions import RequestValidationError
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from starlette.exceptions import HTTPException as StarletteHTTPException


logger = logging.getLogger("app")

_UPLOAD_STATUS = {
    UnsupportedFileTypeError: 415,
    EmptyFileError: 400,
    FileTooLargeError: 413,
    TooManyPagesError: 413,
    MalformedPDFError: 422,
    NoExtractableTextError: 422,
}

_ERROR_RESPONSES = {
    400: {"model": ErrorResponse},
    413: {"model": ErrorResponse},
    415: {"model": ErrorResponse},
    422: {"model": ErrorResponse},
    503: {"model": ErrorResponse},
    504: {"model": ErrorResponse},
}


def _error_response(status_code: int, code: str, message: str) -> JSONResponse:
    return JSONResponse(
        status_code=status_code, content={"error": {"code": code, "message": message}}
    )


def create_app(
    settings: Optional[Settings] = None,
    model_service: Optional[AnswerGenerator] = None,
) -> FastAPI:
    """Application factory. ``settings`` and ``model_service`` can be injected (used by tests)."""
    settings = settings or get_settings()
    logging.basicConfig(
        level=getattr(logging, settings.log_level.upper(), logging.INFO),
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )
    service: AnswerGenerator = model_service or RemoteModelService(settings)

    @asynccontextmanager
    async def lifespan(_: FastAPI):
        # Only the built-in ModelService is loaded here; injected services manage themselves.
        if isinstance(service, ModelService) and settings.load_model_on_startup:
            try:
                await run_in_threadpool(service.load)
            except ModelServiceError as exc:
                # The API still starts so /health can report the problem clearly.
                logger.error("Model not available at startup: %s", exc.message)
        yield

    app = FastAPI(
        title="SLM Document Analysis API",
        description="Send a PDF and a question in one request; a small language model answers "
        "from the document's structure-preserving text. Nothing is stored on the server.",
        version="2.0.0",
        lifespan=lifespan,
    )
    app.state.settings = settings
    app.state.model_service = service

    app.add_middleware(
        CORSMiddleware,
        allow_origins=settings.cors_origin_list,
        allow_methods=["GET", "POST", "OPTIONS"],
        allow_headers=["Content-Type"],
        allow_credentials=False,
    )

    # --- error handlers ------------------------------------------------------
    @app.exception_handler(ApiError)
    async def _api_error(_: Request, exc: ApiError) -> JSONResponse:
        return _error_response(exc.status_code, exc.code, exc.message)

    @app.exception_handler(RequestValidationError)
    async def _validation_error(_: Request, exc: RequestValidationError) -> JSONResponse:
        # Report only field location and a short message; never echo the submitted input.
        problems = "; ".join(
            f"{'.'.join(str(p) for p in e['loc'] if p != 'body')}: {e['msg']}"
            for e in exc.errors()
        )
        return _error_response(422, "validation_error", f"Invalid request: {problems}")

    @app.exception_handler(StarletteHTTPException)
    async def _http_error(_: Request, exc: StarletteHTTPException) -> JSONResponse:
        return _error_response(exc.status_code, "http_error", str(exc.detail))

    @app.exception_handler(Exception)
    async def _unexpected_error(_: Request, exc: Exception) -> JSONResponse:
        logger.error("Unhandled error", exc_info=exc)
        return _error_response(500, "internal_error", "An unexpected error occurred.")

    # --- endpoints -------------------------------------------------------------
    @app.get("/health", response_model=HealthResponse, tags=["system"])
    def health() -> HealthResponse:
        status = service.status
        return HealthResponse(
            status="ok",
            model_ready=bool(status.get("model_ready")),
            model_name=status.get("model_name"),
            adapter_loaded=bool(status.get("adapter_loaded")),
            device=status.get("device"),
            model_error=status.get("load_error"),
        )

    @app.post(
        "/answer",
        response_model=AnswerResponse,
        responses=_ERROR_RESPONSES,
        tags=["question answering"],
    )
    async def answer_question(
        file: UploadFile = File(..., description="A text-based PDF."),
        question: str = Form(..., description="The question about the PDF."),
    ) -> AnswerResponse:
        question = question.strip()
        if not question:
            raise ApiError(422, "empty_question", "The question must not be empty.")
        if len(question) > settings.max_question_chars:
            raise ApiError(
                422,
                "question_too_long",
                f"The question must be at most {settings.max_question_chars} characters.",
            )

        # Fail fast, before spending time on extraction, if the model cannot answer.
        if not service.is_ready:
            status = service.status
            raise ApiError(
                503,
                "model_not_ready",
                status.get("load_error")
                or "The language model is not ready. Check GET /health for details.",
            )

        # Read at most max_bytes + 1 so oversized uploads are detected without buffering them all.
        data = await file.read(settings.max_upload_bytes + 1)
        filename = sanitize_filename(file.filename)
        try:
            structured = await run_in_threadpool(
                process_pdf,
                file.filename,
                data,
                max_bytes=settings.max_upload_bytes,
                min_text_chars=settings.min_text_chars,
                max_pages=settings.max_pages,
            )
        except DocumentProcessingError as exc:
            status_code = next(
                (s for cls, s in _UPLOAD_STATUS.items() if isinstance(exc, cls)), 422
            )
            logger.info("Upload rejected: %s", exc.code)
            raise ApiError(status_code, exc.code, exc.message) from exc
        except Exception as exc:
            logger.exception("Unexpected failure while processing PDF")
            raise ApiError(500, "processing_failed", "The document could not be processed.") from exc

        markdown = structured.markdown
        logger.info(
            "Document extracted pages=%d headings=%d tables=%d chars=%d",
            structured.page_count, structured.heading_count, structured.table_count, len(markdown),
        )

        try:
            started = time.monotonic()
            result = await run_in_threadpool(service.generate_answer, markdown, question)
            elapsed = time.monotonic() - started
        except InferenceTimeoutError as exc:
            raise ApiError(504, exc.code, exc.message) from exc
        except (ModelNotReadyError, ModelNotConfiguredError) as exc:
            raise ApiError(503, exc.code, exc.message) from exc
        except ModelServiceError as exc:
            raise ApiError(500, exc.code, exc.message) from exc

        # Plain string matching of the answer against the extracted elements (not model output).
        locations: list[AnswerLocation] = []
        if not result.unanswerable:
            locations = [
                AnswerLocation(
                    page_number=loc.page_number,
                    element_type=loc.element_type,
                    section_path=list(loc.section_path),
                    snippet=loc.snippet,
                    table_cell=(
                        TableCell(
                            value=loc.table_cell.value,
                            row_label=loc.table_cell.row_label,
                            column_label=loc.table_cell.column_label,
                        )
                        if loc.table_cell
                        else None
                    ),
                )
                for loc in locate_answer(structured.elements, result.answer)
            ]

        limit = settings.max_markdown_response_chars
        message: Optional[str] = None
        if result.unanswerable and result.document_truncated:
            message = (
                "The model found no answer in the beginning of the document, which is all it "
                "could read (about 1-2 pages). The answer may be in the part that was cut off."
            )
        elif result.unanswerable:
            message = "The model judged that the document does not contain the answer."
        elif result.document_truncated:
            message = (
                "The document is longer than the model can read (about 1-2 pages), so only its "
                "beginning was used. The answer may be wrong or missing information."
            )

        return AnswerResponse(
            question=question,
            answer=result.answer,
            unanswerable=result.unanswerable,
            message=message,
            model_name=result.model_name,
            inference_time_seconds=round(elapsed, 3),
            prompt_tokens=result.prompt_tokens,
            generated_tokens=result.generated_tokens,
            document=DocumentInfo(
                filename=filename,
                page_count=structured.page_count,
                heading_count=structured.heading_count,
                table_count=structured.table_count,
                character_count=len(markdown),
                word_count=len(markdown.split()),
                empty_page_numbers=structured.empty_page_numbers,
            ),
            document_truncated=result.document_truncated,
            document_chars_used=result.document_chars_used,
            document_chars_total=result.document_chars_total,
            structured_text=markdown[:limit],
            structured_text_truncated=len(markdown) > limit,
            answer_locations=locations,
        )

    return app



if "*" in ALLOWED_ORIGINS:
    raise ValueError("ALLOWED_ORIGINS must list explicit origins; a wildcard '*' is not allowed.")
if API_HOST not in ("127.0.0.1", "localhost"):
    raise ValueError("API_HOST must be 127.0.0.1 or localhost. The tunnel connects locally.")

app = create_app(settings=settings, model_service=model_service)

print("FastAPI app created.")
print("Allowed browser origins:", settings.cors_origin_list)
print("Routes:", sorted({f"{sorted(r.methods - {'HEAD', 'OPTIONS'})[0]} {r.path}" for r in app.routes if hasattr(r, "methods") and r.path.startswith(("/health", "/answer"))}))


## Cell 8: Start the API server and the tunnel

Colab's `localhost` is **not reachable from your MacBook**, so two things happen here:

1. **uvicorn** runs the API in a background thread on `127.0.0.1:8000` (inside Colab only). It stays up
   while the Colab session is alive, and the cell returns immediately so you can keep using the notebook.
2. **cloudflared** (Cloudflare "quick tunnel") publishes that local port at a temporary
   `https://<random>.trycloudflare.com` address. No account and **no token** are needed, so there is no
   secret to protect or leak. The binary is downloaded from Cloudflare's official GitHub releases.

**Using it**
* Copy the `VITE_API_BASE_URL=...` line printed at the end into the frontend's `.env`, then restart
  `npm run dev`.
* Anyone who knows the random URL can call the API (CORS only restricts *browsers*). Treat the URL as
  private and stop the tunnel when you are done (Cell 8b).
* Cloudflare cuts requests at about 100 s, so keep `GENERATION_TIMEOUT_SECONDS` at 90 or below.
* **Restart:** re-run this cell (it stops the previous server and tunnel first). The URL changes, so
  update the frontend `.env`. If you changed Cells 2 to 7, re-run those first. If the Colab runtime
  was restarted, run everything from Cell 1.

Alternatives such as ngrok need an auth token; if you use one, keep it in Colab Secrets
(`userdata.get(...)`), never in a cell or in the frontend. They are not implemented here.


In [ ]:
import os
import queue
import re
import subprocess
import tempfile
import threading
import time
import urllib.error
import urllib.request

import uvicorn

WORK_DIR = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
CLOUDFLARED_PATH = os.path.join(WORK_DIR, "cloudflared")
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
_TUNNEL_URL = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")

_server = None
_server_thread = None
_tunnel_process = None


def stop_server() -> None:
    global _server, _server_thread
    if _server is not None:
        _server.should_exit = True
    if _server_thread is not None:
        _server_thread.join(timeout=10)
    _server = _server_thread = None


def stop_tunnel() -> None:
    global _tunnel_process
    if _tunnel_process is not None and _tunnel_process.poll() is None:
        _tunnel_process.terminate()
        try:
            _tunnel_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            _tunnel_process.kill()
    _tunnel_process = None


def start_server() -> None:
    """Run uvicorn in a background thread (a notebook cell cannot block)."""
    global _server, _server_thread
    stop_server()
    _server = uvicorn.Server(uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="info"))
    _server_thread = threading.Thread(target=_server.run, name="uvicorn", daemon=True)
    _server_thread.start()
    for _ in range(100):
        if _server.started:
            return
        if not _server_thread.is_alive():
            raise RuntimeError(f"The server failed to start. Is port {API_PORT} already in use? Re-run this cell.")
        time.sleep(0.1)
    raise RuntimeError("The server did not start within 10 seconds.")


def start_tunnel(timeout: float = 60.0) -> str:
    """Start a cloudflared quick tunnel to the local server and return its public https URL."""
    global _tunnel_process
    stop_tunnel()
    if not os.path.exists(CLOUDFLARED_PATH):
        print("Downloading cloudflared ...")
        urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
        os.chmod(CLOUDFLARED_PATH, 0o755)

    proc = subprocess.Popen(
        [CLOUDFLARED_PATH, "tunnel", "--no-autoupdate", "--url", f"http://{API_HOST}:{API_PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    _tunnel_process = proc
    lines: "queue.Queue[str]" = queue.Queue()

    def _pump() -> None:  # keep draining output so the pipe never blocks cloudflared
        for line in proc.stdout:
            lines.put(line)

    threading.Thread(target=_pump, daemon=True).start()

    recent: list[str] = []
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            if proc.poll() is not None:
                break
            continue
        recent.append(line.rstrip())
        match = _TUNNEL_URL.search(line)
        if match and match.group(0) != "https://api.trycloudflare.com":
            return match.group(0)
    stop_tunnel()
    raise RuntimeError("cloudflared did not report a tunnel URL. Last output:\n" + "\n".join(recent[-10:]))


def http_request(url: str, method: str = "GET", headers: dict | None = None, timeout: float = 15.0):
    """Tiny HTTP helper for self-checks. Returns (status, response_headers, body_text)."""
    req = urllib.request.Request(url, method=method, headers={"User-Agent": "slm-backend-selfcheck/1.0", **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, dict(resp.headers), resp.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as err:
        return err.code, dict(err.headers), err.read().decode("utf-8", "replace")


# ---- start the local server and check it ------------------------------------------------
start_server()
LOCAL_URL = f"http://{API_HOST}:{API_PORT}"
status, _, body = http_request(f"{LOCAL_URL}/health")
print(f"\nLocal check  GET {LOCAL_URL}/health -> {status}\n{body}")

# ---- verify CORS from the server's point of view ------------------------------------------
for origin in [*ALLOWED_ORIGINS[:1], "https://not-allowed.example"]:
    status, hdrs, _ = http_request(
        f"{LOCAL_URL}/answer",
        method="OPTIONS",
        headers={"Origin": origin, "Access-Control-Request-Method": "POST"},
    )
    allowed = hdrs.get("access-control-allow-origin")
    print(f"CORS preflight from {origin}: HTTP {status}, allow-origin = {allowed!r}")

# ---- optional public tunnel -------------------------------------------------------------------
if START_TUNNEL:
    PUBLIC_URL = start_tunnel()
    print("\nTunnel started:", PUBLIC_URL)
    for attempt in range(15):  # a brand-new tunnel hostname can take a few seconds to resolve
        try:
            status, _, body = http_request(f"{PUBLIC_URL}/health")
            if status == 200:
                print(f"Tunnel check GET {PUBLIC_URL}/health -> {status}\n{body}")
                break
        except OSError:
            pass
        time.sleep(2)
    else:
        print("WARNING: the tunnel did not answer /health yet. Wait a few seconds and open the URL in a browser.")
    print("\n" + "=" * 78)
    print("Put this line in frontend/.env, then restart `npm run dev`:\n")
    print(f"VITE_API_BASE_URL={PUBLIC_URL}")
    print("=" * 78)
else:
    print("\nSTART_TUNNEL is False: the API is reachable only inside Colab at", LOCAL_URL)


### Cell 8b: Stop the server and the tunnel (optional)

Run this when you are finished. It stops the public URL and the API. (The server never stores
uploaded documents, so there is nothing to clean up.) To start again, re-run Cell 8.


In [ ]:
stop_tunnel()
stop_server()
print("Tunnel and server stopped.")
